In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")


In [ ]:
train_data.isna().sum()

## Taking a quick look at the Data Structure

In [ ]:
train_data.head()

In [ ]:
train_data.info()

`PassengerId` is just the row number and is not useful for model training


In [ ]:
test_data.isna().sum()

***Test data has one missing fare whereas train data does not***

In [ ]:
features = ['Sex', 'Age', 'Pclass', 'Fare', 'SibSp', 'Parch', 'FamilySize', 'Embarked']
#Cabin not included as it will be dealt with separately
#PassengerId is just the index of the row
#Name is a string we will extract the titles of individual names and then deal with it
#Ticket is a string which generated just for unique identification of passenger tickets
#FamilySize = Parch + SibSp

# Relationship between key attributes
## 1. `Survived` vs `Sex`
> This plot shows that females were given priority in the survival efforts.

In [ ]:
ax = sns.countplot(x='Sex', hue='Survived', data=train_data)
plt.title("Survival by Gender")
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', fontsize=10, padding=3)

plt.show()

## 2. `Pclass` vs `Fare`
* Since the ship would have all kinds of passengers it would be imperative to classify them into groups according to fares they paid like in the modern tavelling system (first classs , bussines class, economic class,ect.)

In [ ]:
sns.scatterplot(data=train_data,x='Fare',y='Pclass',hue='Survived',alpha=0.7,palette='Set1')
plt.title("Passenger Class vs Fare (colored by Survival)")
plt.show()

>This shows that the `Pclass`= 1 were the passengers who pent their most on the tickets for the cruise followed by 2 and 3.
>
>It should be noted that `Pclass` =2,3 have paid similiarly for the tickets which implies that the `Fare` isn't the only determining attribute for `Pclass`

## 3. `Survived` vs `Pclass`

In [ ]:
Pclass_rate = train_data.groupby('Pclass')['Survived'].mean()
Pclass_rate

In [ ]:
ax=sns.countplot(x='Pclass',hue='Survived',data=train_data)
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', fontsize=10, padding=3)

plt.show()

> This plot indicates that passengers in `Pclass` = 1 and 2 were given priority in survival efforts
>
> This implies that the rich were given priority in the survival efforts or they had better positioning of their cabins in order to escape the ship.
>
> Also the no. of `Survived` = 1 is comparable for all the classes. Could it mean the number of lifeboats alloted for each Passenger Class was same ?

## 4. `Age` Distribution in survival overlay

In [ ]:
ax = sns.histplot(data=train_data, x='Age', hue='Survived', kde=True)
plt.title("Age Distribution by Survival")
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', fontsize=10, padding=3)

plt.show()


> The distribution is somewhat similiar to a gaussian distribution with a left skew.
> 
> Children are always the no.1 on the priority list in survival efforts which is indicated here as well

## 5. `Fsize` vs `Survival`
   * `Fsize` = Total number of dependents on a single Passenger
   

In [ ]:
# Compute family size temporarily
Fsize = train_data["SibSp"] + train_data["Parch"]

# Group Survived values by family size
survived_by_Fsize = train_data.groupby(Fsize)["Survived"].mean() # replcae sum() with mean() for rate

# Plot histogram (or bar chart — bar chart is clearer here)
plt.figure(figsize=(10, 6))
plt.bar(survived_by_Fsize.index, survived_by_Fsize.values, color='skyblue', edgecolor='black')
plt.xlabel("Family Size (SibSp + Parch)")
plt.ylabel("Survival")
plt.title("Survival by Family Size")
plt.xticks(survived_by_Fsize.index)
plt.show()

> The number of people surviving if travelling alone is more compared to other groups
>
> But when loking at the rate of survival in each groups the survival rate is much more in the `Fsize` = `1` to `3`
>
> Survival is more likely when combining the efforts of other family members, but **Too many cooks spoil the broth**? As it can be seen that Passengers with `Fsize`>3 have lower rate of survival.

We will bin the `Fsize` into 3 categories namely - Solo(0), Small(1-3), Large(4-10)

## 6. Embarked vs Survived

In [ ]:
df_corr = train_data[['Embarked', 'Survived']]
df_corr_ohe = pd.get_dummies(df_corr, columns=['Embarked'])
corr_matrix = df_corr_ohe.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(
    corr_matrix,
    annot=True,          # display correlation values
    cmap='coolwarm',     # nice diverging color map
    fmt=".2f",           # format decimals
    linewidths=0.5
)
plt.title("Correlation Matrix for Numerical Attributes")
plt.show()

### Lets plot the Correlation Matrix for the atrributes

In [ ]:
num_cols = train_data.select_dtypes(include=['number'])

num_cols.head()

In [ ]:
corr_matrix = num_cols.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(
    corr_matrix,
    annot=True,          # display correlation values
    cmap='coolwarm',     # nice diverging color map
    fmt=".2f",           # format decimals
    linewidths=0.5
)
plt.title("Correlation Matrix for Numerical Attributes")
plt.show()

1. `Pclass` is inversely related to `Fare` i.e. `Pclass`= 1 paid more for the ticket
2. `Age` is inversely relatedv to `Survived` i.e. younger people had higher chance of survival.
3. `Parch` is directly related to `Fare` i.e. people with parents onboard paid more for the comfort of their parents.
4. `Pclass` is inversly related to `Age` i.e. younger people chose lower classes.
5. `Age` is inversely related to both `SibSp` and `Parch` i.e. younger people have more siblings and parents on board.

# Feature Engineering

There are various attributes that are not useful to us in the form they are presented.
1. `Name` is unique to each passenger but their titles can be extracted from it, titles indicate their importance.
2. `Cabin` has the most missing values in it. Also it contains the cabin values of the passengers in the string form. The Cabin Types can be extracted from it and one-hot encoded.
3. As mentioned earlier `Fsize` could be binned and then `SibSp` and `Parch` would be dropped when training the model.
4. `Fare` is distributed vastly we can Log-transform it (x-> log(1+x)). This would not destablise our training models

In [ ]:
import re
from sklearn.base import BaseEstimator, TransformerMixin

class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self,name_col='Name',cabin_col='Cabin',sibsp_col='SibSp',parch_col='Parch',fare_col='Fare',age_col='Age',embarked_col='Embarked',drop_cols=['Name','Cabin', 'Ticket', 'PassengerId']):
        self.name_col = name_col
        self.cabin_col = cabin_col
        self.sibsp_col = sibsp_col
        self.parch_col = parch_col
        self.fare_col = fare_col
        self.age_col = age_col
        self.drop_cols = drop_cols
        self.embarked_col = embarked_col

    def fit(self, X, y=None):
        # no fitting needed for deterministic transformations
        self.age_mean = X[self.age_col].mean()
        self.age_std = X[self.age_col].std()
        self.fare_mean = X[self.fare_col].mean()
        self.embarked_mode = X[self.embarked_col].mode()[0]
        return self

    def transform(self, X):
        X_copy = X.copy()

        # --- 1. Extract Title from Name ---
        X_copy['Title'] = X_copy[self.name_col].apply(
            lambda name: re.search(r' ([A-Za-z]+)\.', name).group(1)
        )

        # Group rare titles to avoid sparse matrix explosion
        rare_titles = ['Dr','Rev','Col','Major','Mlle','Countess','Ms','Lady','Jonkheer',
                       'Don','Dona','Mme','Capt','Sir']
        X_copy['Title'] = X_copy['Title'].replace({
            'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'
        })
        X_copy['Title'] = X_copy['Title'].apply(
            lambda t: 'Rare' if t in rare_titles else t
        )

        # --- 2. Extract CabinType ---
        X_copy['CabinCount'] = X_copy[self.cabin_col].fillna('Missing').apply(lambda c: len(c.split()) if c!= 'Missing' else 0)
        X_copy['CabinType'] = X_copy[self.cabin_col].fillna('Missing').apply(lambda c: c[0] if c != 'Missing' else 'Missing')
        

        # --- 3. Create Family Size Bin ---
        X_copy['Fsize'] = X_copy[self.sibsp_col] + X_copy[self.parch_col] + 1
        def fsize_bin(f):
            if f == 1:
                return 'Single'
            elif f <= 4:
                return 'Small'
            else:
                return 'Large'
        X_copy['FsizeBin'] = X_copy['Fsize'].apply(fsize_bin)

        # Drop SibSp and Parch 
        #X_copy = X_copy.drop(columns=[self.sibsp_col, self.parch_col, 'Fsize'])

        # --- 4. Log Transform Fare ---
        X_copy[self.fare_col] = X_copy[self.fare_col].fillna(self.fare_mean).apply(lambda x: np.log1p(x))

        #--- 5. Standardize Age ---
        X_copy[self.age_col] = X_copy[self.age_col].fillna(self.age_mean)
        X_copy[self.age_col] = X_copy[self.age_col].apply(lambda x: (x-self.age_mean)/self.age_std)

        #--- 6. Embarked Imputation ---
        X_copy[self.embarked_col] = X_copy[self.embarked_col].fillna(self.embarked_mode)

        #--- Dropping useless coloumns ---
        X_copy.drop(columns=self.drop_cols, inplace=True, errors='ignore')

        
        return X_copy

#let us transform the training data into the preprocessed data that will work with the model

transformer = TitanicFeatureEngineer()
titanic_transformed = transformer.fit_transform(train_data)
titanic_transformed.head()

In [ ]:
titanic_transformed.isna().sum()

# Training Models

Since this is a case of binary classification we have multiple options to choose from for training our model
1. Base model - Logistic Regression 
2. Tree model - Random Forest
3. Kernel - Support Vector Classifier
4. Ensemble - XGBoost

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

# 1 Apply custom feature engineering
# 2 Encode categorical features
# 3 Fit model

categorical_features = ['Sex', 'Embarked', 'Title', 'CabinType', 'FsizeBin']
numeric_features = ['Pclass', 'Fare', 'Age', 'CabinCount','SibSp','Parch','Fsize']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', 'passthrough', numeric_features)
])

base_pipeline = Pipeline([
    ('features', TitanicFeatureEngineer()),
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=5000,random_state=42))
])

models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}


base_pipeline.fit(train_data, train_data['Survived'])



## 1.Training Logistic Ression model

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_base = [
    {
        'clf__penalty': ['l2'],
        'clf__C': [0.01, 0.1, 1, 10, 100],
        'clf__solver': ['lbfgs', 'saga']
    },
    {
        'clf__penalty': ['l1'],
        'clf__C': [0.01, 0.1, 1, 10, 100],
        'clf__solver': ['saga']
    },
    {
        'clf__penalty': ['elasticnet'],
        'clf__C': [0.01, 0.1, 1, 10, 100],
        'clf__solver': ['saga'],
        'clf__l1_ratio': [0, 0.5, 1]
    }
]

grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid_base,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=0
)

X = train_data.drop('Survived', axis=1)
y = train_data['Survived']

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    grid_search.fit(X, y)

print("Best Params:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

### Comparing Tuned baseline vs other models(default parameters)

In [ ]:


from sklearn.model_selection import cross_val_score

baseline_acc = grid_search.best_score_
results = {"Logistic Regression (Baseline)": baseline_acc}

for name, model in models.items():
    pipeline = Pipeline([
        ('features', TitanicFeatureEngineer()),
        ('preprocess', preprocessor),
        ('clf', model)
    ])
    
    scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
    results[name] = np.mean(scores)

print("\nModel Comparison (5-Fold CV Accuracy):")
for model_name, acc in results.items():
    print(f"{model_name}: {acc:.4f}")


> ***SVC and XGBoost show promising accuaracy even without hyperparameter tuning***

## 2. Training Support Vector Machine

In [ ]:
param_grid_svc = {
    "clf__C": [0.5, 0.8,0.9, 1, 1.2, 1.5, 2],
    "clf__gamma": ["auto", "scale", 0.01, 0.05, 0.1],
    "clf__kernel": ["rbf","linear","poly"],
    "clf__class_weight": [None, "balanced"]
}

svc_pipeline = Pipeline([
    ('features', TitanicFeatureEngineer()),
    ('preprocess', preprocessor),
    ('clf', SVC(probability=True, random_state=42))
])

grid_search = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=param_grid_svc,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=0
)

X = train_data.drop('Survived', axis=1)
y = train_data['Survived']

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    grid_search.fit(X, y)

best_svc_estimator = grid_search.best_estimator_

print("Best Params:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

param_grid_xgb = {
    "classifier__n_estimators": [100, 200, 500],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__subsample": [0.7, 1.0],
    "classifier__colsample_bytree": [0.7, 1.0]
}

## 3. Training and Tuning XGBoost model

In [ ]:
xgb_pipeline = Pipeline([
    ('features', TitanicFeatureEngineer()),
    ('preprocess', preprocessor),
    ('clf', XGBClassifier(random_state=42, eval_metric='logloss'))
])


param_grid_xgb = {
    "clf__n_estimators": [100, 200, 500],
    "clf__max_depth": [3, 5, 7],
    "clf__learning_rate": [0.01, 0.1, 0.2],
    "clf__subsample": [0.7, 1.0],
    "clf__colsample_bytree": [0.7, 1.0]
}
grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid_xgb,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=0
)

X = train_data.drop('Survived', axis=1)
y = train_data['Survived']

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    grid_search.fit(X, y)

best_xgb_estimator = grid_search.best_estimator_

print("Best Params:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

### Checking the Coffusion matrix and Precission-Recall Curve for the SVC, Logistic and XGBoost Models

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# 1. P-R Curve for SVC Model
y_scores = best_svc_estimator.decision_function(X)
precision, recall, thresholds = precision_recall_curve(y, y_scores)
avg_precision = average_precision_score(y, y_scores)
plt.plot(recall, precision, label=f'PR curve (AP = {avg_precision:.2f})')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve for Best SVC Model")
plt.legend()
plt.grid(True)
plt.show()

# 2. P-R Curve for XGBoost Model
y_scores = best_xgb_estimator.predict_proba(X)[:, 1]
precision, recall, thresholds = precision_recall_curve(y, y_scores)
avg_precision = average_precision_score(y, y_scores)
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, color='blue', lw=2,label=f'XGBoost (AP = {avg_precision:.3f})')
plt.fill_between(recall, precision, alpha=0.2, color='blue')
plt.title('Precision–Recall Curve for XGBoost')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='lower left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
import joblib
joblib.dump(grid_search, 'grid_search_full.pkl')

In [ ]:

print("Best accuracy:", grid_search.best_score_)
print("Best parameters:", grid_search.best_params_)
best_model = grid_search.best_estimator_

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Compute confusion matrix
y_pred = best_model.predict(X)
cm = confusion_matrix(y, y_pred)

# Plot it
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

# y_score = predicted probability for the positive class
y_score = best_model.predict_proba(X)[:, 1]

RocCurveDisplay.from_predictions(y, y_score)
plt.title("ROC Curve")
plt.show()

In [ ]:
# Getting Feature importances to better understande the behaviour
feature_names = best_xgb_estimator.named_steps['preprocess'].get_feature_names_out()
xgb_model = best_xgb_estimator.named_steps['clf']

importances = xgb_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(importance_df.head(15))

In [ ]:
#FINALLY choosing XGBoosted model
X_test = test_data.copy()
#best_model.fit(X_test)
y_pred = best_xgb_estimator.predict(X_test)
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': y_pred
})

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': y_pred
})
submission.to_csv("submission.csv", index=False)